In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, year, lit
from pyspark.sql.types import IntegerType, DateType

# 1. إنشاء جلسة Spark
spark = SparkSession.builder \
    .appName("SoccerDataPreparation") \
    .getOrCreate()

# 2. تحميل البيانات من ملف CSV
# ملاحظة: تأكد من المسار الصحيح للملف داخل الحاوية
input_path = "../data/rim_championnat_results_2007-2025.csv"
df = spark.read.csv(input_path, header=True, inferSchema=True)

print("Original Schema:")
df.printSchema()
df.show(5)

# 3. تنظيف البيانات (Data Cleaning)
# أ- تحويل الأعمدة إلى أنواعها الصحيحة (خاصة الأهداف والتواريخ)
# ب- التعامل مع القيم المفقودة (سنقوم بحذف الصفوف التي بها قيم مفقودة في الأعمدة الأساسية)
df_clean = df.dropna(subset=["date", "home_team", "away_team", "home_goals", "away_goals"]) \
             .withColumn("home_goals", col("home_goals").cast(IntegerType())) \
             .withColumn("away_goals", col("away_goals").cast(IntegerType())) \
             .withColumn("date", col("date").cast(DateType()))

# 4. إنشاء الأعمدة المشتقة (Derived Columns) - حسب المطلوب في المشروع
df_enriched = df_clean \
    .withColumn("total_goals", col("home_goals") + col("away_goals")) \
    .withColumn("goal_difference", col("home_goals") - col("away_goals")) \
    .withColumn("result", 
        when(col("home_goals") > col("away_goals"), "home_win")
        .when(col("home_goals") < col("away_goals"), "away_win")
        .otherwise("draw")) \
    .withColumn("match_year", year(col("date")))

# 5. عرض النتائج النهائية للتأكد
print("Enriched Schema:")
df_enriched.printSchema()
df_enriched.show(5)

# 6. حفظ البيانات بصيغة Parquet
output_path = "outputs/prepared/cleaned_soccer_data.parquet"
df_enriched.write.mode("overwrite").parquet(output_path)

print(f"Success! Data saved to {output_path}")